# Episode 28: Attention heads in our GPT, seen and tested

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep028-attention.ipynb)

We look inside our agent while it copies a name into a query, with each head's attention written out, the logit lens, and a page of our own, then turn the heads that look at the name off to see what they do.

[The video](https://youtu.be/Lq4id0KSNWk) · [The episode's page](https://mvgpt.nenuadrian.com/ep028-attention/) · [The docs](https://nenuadrian.github.io/mvGPT/attention/) · [The code at `ep028/end`](https://github.com/nenuadrian/mvGPT/tree/ep028/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep028/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


import shlex
import subprocess
import time
import urllib.error
import urllib.parse
import urllib.request


def start(command, url):
    # A server in the background, once it answers at url; its output goes to /tmp/server-<port>.log.
    log = open(f"/tmp/server-{urllib.parse.urlparse(url).port}.log", "w")
    server = subprocess.Popen(shlex.split(command), stdout=log, stderr=subprocess.STDOUT)
    while True:
        try:
            urllib.request.urlopen(url, timeout=1)
            return server
        except urllib.error.HTTPError:
            return server
        except OSError:
            if server.poll() is not None:
                raise RuntimeError(open(log.name).read())
            time.sleep(1)


def stop(server):
    server.terminate()
    server.wait()


fetch("graph-agent")

## The tests
The weights, the same logits as PyTorch's attention, the lens's last layer, and heads turned off and back on.

In [ ]:
!python -m pytest -q tests/test_interpret.py

## The logit lens
Episode 25's agent, from the Hub: each layer's three likeliest next tokens as the graph role
opens the name in its query, for the King Penguin, the Okapi, and the Okapi after the `O`. A
bash cell, as `!` would read the query's braces as Python.

In [ ]:
%%bash
python interpret.py checkpoints/graph-agent/model.pt lens "How much does the King Penguin weigh?" " <<cypher MATCH (s:Species {name: '"
python interpret.py checkpoints/graph-agent/model.pt lens "How much does the Okapi weigh?" " <<cypher MATCH (s:Species {name: '"
python interpret.py checkpoints/graph-agent/model.pt lens "How much does the Okapi weigh?" " <<cypher MATCH (s:Species {name: 'O"

## smollerGraph
The graph the graph role asks, as in episode 25, with `sg serve` on port 8474, where mvGPT
looks for it, and episode 24's questions.

In [ ]:
if not os.path.isdir("/content/smollerGraph"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep010/end https://github.com/nenuadrian/smollerGraph /content/smollerGraph
!pip install --quiet cmake ninja
!cmake -S /content/smollerGraph -B /content/smollerGraph/build -G Ninja > /dev/null && cmake --build /content/smollerGraph/build --target sg -- --quiet
!rm -f /content/smollerGraph/wikidata.sg && /content/smollerGraph/build/sg load /content/smollerGraph/wikidata.sg /content/smollerGraph/data/wikidata.txt
sg = start("/content/smollerGraph/build/sg serve /content/smollerGraph/wikidata.sg 8474", "http://127.0.0.1:8474/api/names?prefix=R")

In [ ]:
!python graph_data.py /content/smollerGraph/data/wikidata.txt

## Heads turned off
Each head's share of its attention on the name in the task, over episode 24's held-out
questions, then the graph role's questions with the top heads turned off, and with as many
random ones.

In [ ]:
!python interpret.py checkpoints/graph-agent/model.pt heads

## The page
`viewer.py` writes a page for a prompt and an answer, with the heads' scores from above: pick
a token, a head or a whole layer. The King Penguin's query, then the Okapi's as the model wrote
it.

In [ ]:
%%bash
python viewer.py checkpoints/graph-agent/model.pt "How much does the King Penguin weigh?" " <<cypher MATCH (s:Species {name: 'King Penguin'}) RETURN s.weight_g>>" --heads results/heads-graph-agent.json --out king.html
python viewer.py checkpoints/graph-agent/model.pt "How much does the Okapi weigh?" " <<cypher MATCH (s:Species {name: 'Oakapi'}) RETURN s.weight_g>>" --heads results/heads-graph-agent.json --out okapi.html

In [ ]:
from IPython.display import HTML, display

display(HTML(filename="king.html"))

In [ ]:
display(HTML(filename="okapi.html"))